In [0]:
from pyspark.sql import functions as F

# Leer df_final desde la tabla Delta registrada en Unity Catalog (guardada en la Actividad 02)
df_final = spark.table("workspace.default.financial_final_daniel")

print(f"Filas: {df_final.count():,} | Columnas: {len(df_final.columns)}")
df_final.printSchema()

In [0]:
df = df_final \
    .withColumn("hora", F.hour("transaction_date")) \
    .withColumn("dia_semana", F.dayofweek("transaction_date")) \
    .withColumn("mes", F.month("transaction_date")) \
    .withColumn("anio", F.year("transaction_date")) \
    .withColumn("semana_anio", F.weekofyear("transaction_date")) \
    .withColumn("dia_mes", F.dayofmonth("transaction_date"))

df.select("transaction_date", "hora", "dia_semana", "mes", "semana_anio").show(5)

In [0]:
# Agrupar transacciones por semana y mes
df = df \
    .withColumn("semana_inicio", F.date_trunc("week", "transaction_date")) \
    .withColumn("mes_inicio", F.date_trunc("month", "transaction_date"))

# Volumen semanal
df.groupBy("semana_inicio") \
    .agg(
        F.count("*").alias("num_transacciones"),
        F.sum("amount").alias("monto_total")
    ) \
    .orderBy("semana_inicio") \
    .show(10)

In [0]:
# ¿Cuántos días han pasado desde la primera transacción del dataset?
fecha_min = df.agg(F.min("transaction_date")).collect()[0][0]

df = df.withColumn(
    "dias_desde_inicio",
    F.datediff(F.col("transaction_date"), F.lit(fecha_min))
)

# ¿Cuántos días entre la transacción y el final del mes?
df = df.withColumn(
    "dias_para_fin_de_mes",
    F.datediff(
        F.last_day(F.col("transaction_date")),
        F.col("transaction_date")
    )
)

df.select("transaction_date", "dias_desde_inicio", "dias_para_fin_de_mes").show(5)

In [0]:
df_mensual_usuario = df.groupBy("user_id", "mes_inicio") \
    .agg(
        F.count("*").alias("num_transacciones"),
        F.sum("amount").alias("gasto_mensual"),
        F.avg("amount").alias("ticket_promedio"),
        F.max("amount").alias("compra_maxima"),
        F.countDistinct("description").alias("categorias_distintas"),
        F.percentile_approx("amount", 0.5).alias("mediana_gasto")
    ) \
    .orderBy("mes_inicio", F.col("gasto_mensual").desc())

df_mensual_usuario.show(10)

In [0]:
# Pivot: gasto total por tipo de tarjeta por mes

df_pivot = (
    df
    .groupBy("mes_inicio")
    .pivot("card_type")
    .agg(F.round(F.sum("amount"), 2))
    .orderBy("mes_inicio")
)

display(df_pivot)

## Pivot — gasto por tipo de tarjeta

Se usó `pivot()` para convertir los valores de `card_type` en columnas y comparar el gasto total mensual por tipo de tarjeta.

El `pivot` tiene sentido cuando se quiere comparar pocas categorías conocidas, como `Debit`, `Credit` o `Debit (Prepaid)`, en una vista más ancha y fácil de leer.

Puede volverse un antipatrón cuando la columna tiene demasiados valores distintos, porque genera muchas columnas, aumenta el costo de procesamiento y puede hacer que la tabla sea difícil de mantener o analizar.

In [0]:
from pyspark.sql import Window

# Crear la ventana de clasificación por categoría
windowSpec = Window.partitionBy("description").orderBy(F.col("gasto_total").desc())

# Sumar el gasto total por comercio
df_por_comercio = df.groupBy("description") \
    .agg(F.sum("amount").alias("gasto_total"))

# Calcular el ranking dentro de cada categoría
df_ranking = df_por_comercio.withColumn("rank_en_categoria", F.rank().over(windowSpec))

# Mostrar el top 3 comercios por categoría
df_ranking.filter(F.col("rank_en_categoria") <= 3) \
    .orderBy("description", "rank_en_categoria") \
    .show(20, truncate=False)

In [0]:
# Ejemplo propio para comparar rank, dense_rank y row_number

datos_ranking = [
    ("A", "Comercio 1", 1000),
    ("A", "Comercio 2", 800),
    ("A", "Comercio 3", 800),
    ("A", "Comercio 4", 600),
]

df_ejemplo_ranking = spark.createDataFrame(
    datos_ranking,
    ["categoria", "comercio", "ventas"]
)

window_ejemplo = Window.partitionBy("categoria").orderBy(F.col("ventas").desc())

df_ejemplo_ranking = (
    df_ejemplo_ranking
    .withColumn("row_number", F.row_number().over(window_ejemplo))
    .withColumn("rank", F.rank().over(window_ejemplo))
    .withColumn("dense_rank", F.dense_rank().over(window_ejemplo))
)

display(df_ejemplo_ranking)

## Diferencia entre row_number, rank y dense_rank

`row_number()` asigna un número único a cada fila dentro de la ventana. Aunque dos filas tengan el mismo valor, cada una recibe una posición diferente.

`rank()` asigna la misma posición a valores empatados, pero deja saltos en el ranking. Por ejemplo, si dos comercios empatan en el puesto 2, el siguiente queda en puesto 4.

`dense_rank()` también asigna la misma posición a valores empatados, pero no deja saltos. Si dos comercios empatan en el puesto 2, el siguiente queda en puesto 3.

Estas funciones son útiles cuando se necesita ordenar registros dentro de grupos sin reducir el número de filas como ocurre con `groupBy`.

In [0]:
# Ranking corregido: comercio con mayor volumen dentro de cada categoría MCC

df_por_comercio_categoria = (
    df
    .groupBy("description", "merchant_id", "merchant_city")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.round(F.sum("amount"), 2).alias("gasto_total")
    )
)

window_categoria = (
    Window
    .partitionBy("description")
    .orderBy(F.col("gasto_total").desc())
)

df_ranking_categoria = (
    df_por_comercio_categoria
    .withColumn("rank_en_categoria", F.rank().over(window_categoria))
)

display(
    df_ranking_categoria
    .filter(F.col("rank_en_categoria") <= 3)
    .orderBy("description", "rank_en_categoria")
)

## Ranking por comercio dentro de categoría

Se calculó el volumen total por `description`, `merchant_id` y `merchant_city`.

Luego se usó `rank()` con una ventana particionada por `description`, para identificar los comercios con mayor volumen dentro de cada categoría MCC.

Esta versión es más completa que agrupar únicamente por `description`, porque permite comparar comercios dentro de cada categoría y no solo el total de la categoría.

In [0]:
# Dentro de cada usuario, ¿cuánto gastó en la transacción anterior?
windowSpec_usuario = Window.partitionBy("user_id").orderBy("transaction_date")

df = df.withColumn("gasto_anterior", F.lag("amount", 1).over(windowSpec_usuario)) \
       .withColumn("gasto_siguiente", F.lead("amount", 1).over(windowSpec_usuario)) \
       .withColumn(
           "variacion_vs_anterior",
           F.round(F.col("amount") - F.col("gasto_anterior"), 2)
       )

# Transacciones donde el monto casi duplica el anterior (posible anomalía)
df.filter(F.col("amount") > F.col("gasto_anterior") * 2) \
  .select("user_id", "transaction_date", "amount", "gasto_anterior", "variacion_vs_anterior") \
  .orderBy(F.col("variacion_vs_anterior").desc()) \
  .show(10)

In [0]:
# Gasto acumulado por usuario ordenado por fecha
windowSpec_acum = Window.partitionBy("user_id") \
    .orderBy("transaction_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df = df.withColumn("gasto_acumulado", F.sum("amount").over(windowSpec_acum))

df.select("user_id", "transaction_date", "amount", "gasto_acumulado") \
  .filter(F.col("user_id") == df.limit(1).select("user_id").collect()[0][0]) \
  .show(10)

In [0]:
# Media móvil de los últimos 3 transacciones por usuario
windowSpec_rolling = Window.partitionBy("user_id") \
    .orderBy("transaction_date") \
    .rowsBetween(-2, 0)  # fila actual y las 2 anteriores

df = df.withColumn("media_movil_3", F.round(F.avg("amount").over(windowSpec_rolling), 2))

df.select("user_id", "transaction_date", "amount", "media_movil_3") \
  .filter(F.col("user_id") == df.limit(1).select("user_id").collect()[0][0]) \
  .show(10)

In [0]:
# Agrupar por hora y contar las transacciones fraudulentas
df_hora_fraude = df.filter(F.col("is_fraud") == "Yes") \
    .groupBy("hora") \
    .agg(F.count("*").alias("num_transacciones_fraudulentas")) \
    .orderBy(F.col("num_transacciones_fraudulentas").desc())

# Mostrar la hora con mayor número de transacciones fraudulentas
df_hora_fraude.show(1)

In [0]:
# Pregunta 1: Hora pico del fraude

df_hora_fraude = (
    df
    .filter(F.col("is_fraud") == "Yes")
    .groupBy("hora")
    .agg(
        F.count("*").alias("num_fraudes")
    )
    .orderBy(F.col("num_fraudes").desc())
)

display(df_hora_fraude)

# Obtener hora con más fraude
hora_mas_fraude = df_hora_fraude.orderBy(F.col("num_fraudes").desc()).first()

# Obtener hora con menos fraude
hora_menos_fraude = df_hora_fraude.orderBy(F.col("num_fraudes").asc()).first()

print(f"Hora con más fraude: {hora_mas_fraude['hora']} con {hora_mas_fraude['num_fraudes']} fraudes")
print(f"Hora con menos fraude: {hora_menos_fraude['hora']} con {hora_menos_fraude['num_fraudes']} fraudes")

## Pregunta 1 — Hora pico del fraude

Se filtraron únicamente las transacciones marcadas como fraudulentas (`is_fraud = Yes`) y luego se agruparon por la columna `hora`.

La hora con más transacciones fraudulentas fue la hora **11**, con **1,610** fraudes.

La hora con menos transacciones fraudulentas fue la hora **22**, con **13** fraudes.

Este análisis permite identificar en qué momento del día se concentra más actividad fraudulenta. En un caso real, esta información podría servir para reforzar reglas de monitoreo o alertas en determinadas franjas horarias.

In [0]:
# Pregunta 2: Efecto fin de semana

df_fin_semana = (
    df
    .withColumn(
        "es_fin_semana",
        F.when(F.col("dia_semana").isin([1, 7]), True).otherwise(False)
    )
    .groupBy("es_fin_semana")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.round(F.avg("amount"), 2).alias("monto_promedio"),
        F.round(F.percentile_approx("amount", 0.5), 2).alias("mediana_monto")
    )
    .orderBy(F.col("monto_promedio").desc())
)

display(df_fin_semana)

## Pregunta 2 — Efecto fin de semana

Se creó la columna `es_fin_semana` usando `dia_semana`, considerando domingo como `1` y sábado como `7`.

El monto promedio fue ligeramente mayor durante fines de semana.

- Fin de semana: **43.10**
- Entre semana: **42.93**

También la mediana fue un poco mayor en fines de semana:

- Fin de semana: **29.07**
- Entre semana: **28.96**

La diferencia no es muy grande, pero el resultado sugiere que durante fines de semana las transacciones tienden a tener un monto promedio apenas superior.

In [0]:
# Pregunta 3: Usuarios cuya última transacción del mes duplica su mediana histórica

from pyspark.sql import Window

# Mediana histórica por usuario
df_mediana_usuario = (
    df
    .groupBy("user_id")
    .agg(
        F.percentile_approx("amount", 0.5).alias("mediana_historica")
    )
)

# Ranking de transacciones dentro de cada usuario y mes, dejando la última primero
window_ultima_tx_mes = (
    Window
    .partitionBy("user_id", "mes_inicio")
    .orderBy(F.col("transaction_date").desc())
)

df_ultima_tx_mes = (
    df
    .withColumn("rn_ultima_tx_mes", F.row_number().over(window_ultima_tx_mes))
    .filter(F.col("rn_ultima_tx_mes") == 1)
)

# Comparar última transacción del mes contra mediana histórica del usuario
usuarios_explotan_gasto = (
    df_ultima_tx_mes
    .join(df_mediana_usuario, on="user_id", how="left")
    .withColumn(
        "duplica_mediana",
        F.col("amount") > (F.col("mediana_historica") * 2)
    )
    .filter(F.col("duplica_mediana") == True)
    .select(
        "user_id",
        "mes_inicio",
        "transaction_date",
        "amount",
        "mediana_historica",
        "duplica_mediana"
    )
    .orderBy(F.col("amount").desc())
)

display(usuarios_explotan_gasto.limit(20))

print(f"Usuarios/mes donde la última transacción duplica la mediana histórica: {usuarios_explotan_gasto.count():,}")

## Pregunta 3 — Usuarios que “explotan” en gasto

Se identificó la última transacción de cada usuario en cada mes usando `row_number()` con una ventana particionada por `user_id` y `mes_inicio`, ordenada por `transaction_date` descendente.

Luego se calculó la mediana histórica de gasto por usuario y se comparó contra el monto de esa última transacción mensual.

Se encontraron **37,027 casos usuario/mes** donde la última transacción del mes duplicó la mediana histórica del usuario.

Este tipo de análisis puede servir para detectar comportamientos atípicos o posibles cambios bruscos en el patrón de consumo de un cliente.

In [0]:
# Pregunta 4: Comercio top por mes usando window functions

df_comercio_mes = (
    df
    .groupBy("mes_inicio", "merchant_id", "merchant_city", "description")
    .agg(
        F.count("*").alias("num_transacciones"),
        F.round(F.sum("amount"), 2).alias("volumen_total")
    )
)

window_comercio_mes = (
    Window
    .partitionBy("mes_inicio")
    .orderBy(F.col("volumen_total").desc())
)

df_comercio_top_mes = (
    df_comercio_mes
    .withColumn("rank_mes", F.rank().over(window_comercio_mes))
    .filter(F.col("rank_mes") == 1)
    .orderBy("mes_inicio")
)

display(df_comercio_top_mes)

## Pregunta 4 — Comercio top por mes

Se agrupó la información por `mes_inicio`, `merchant_id`, `merchant_city` y `description`, calculando el número de transacciones y el volumen total.

Luego se aplicó una window function con `rank()` particionando por `mes_inicio` y ordenando por `volumen_total` descendente.

El resultado muestra que el comercio con mayor volumen mensual aparece de forma recurrente como:

- `merchant_id`: **39021**
- Ciudad: **ONLINE**
- Categoría: **Tolls and Bridge Fees**

Este análisis permite identificar el comercio dominante por volumen en cada mes sin perder el detalle del ranking mensual.

In [0]:
# Pregunta 5: Patrón temporal de fraude a lo largo del año

df_fraude_anual = (
    df
    .groupBy("anio")
    .agg(
        F.count("*").alias("total_transacciones"),
        F.sum(F.when(F.col("is_fraud") == "Yes", 1).otherwise(0)).alias("total_fraudes"),
        F.round(
            F.sum(F.when(F.col("is_fraud") == "Yes", 1).otherwise(0)) / F.count("*"),
            6
        ).alias("tasa_fraude")
    )
    .orderBy("anio")
)

display(df_fraude_anual)

## Pregunta 5 — Patrón temporal de fraude

Se agruparon las transacciones por año y se calculó el total de transacciones, el total de fraudes y la tasa de fraude anual.

El resultado muestra que el fraude no sigue una tendencia completamente creciente o decreciente. Hay años con picos claros y otros con niveles mucho más bajos.

El año con mayor tasa de fraude fue **2010**, con una tasa aproximada de **0.002074**.

El año con menor tasa de fraude fue **2011**, con una tasa aproximada de **0.000029**.

También se observan tasas relativamente altas en **2015** y **2016**. Esto sugiere que el fraude varía por periodo y que sería útil analizar factores adicionales como tipo de tarjeta, categoría de comercio, canal de transacción o ciudad.

## Validación de window functions

En esta actividad se usaron varias window functions:

- `rank()` para ordenar comercios dentro de una categoría o mes.
- `lag()` y `lead()` para comparar una transacción contra la anterior o siguiente.
- `sum().over()` para calcular gasto acumulado por usuario.
- `avg().over()` para calcular media móvil de las últimas tres transacciones.

Estas funciones permiten analizar filas relacionadas sin reducir el número de registros como ocurre con `groupBy`.

## Cierre del análisis avanzado

El análisis final combinó funciones de fecha, agregaciones avanzadas y window functions para responder preguntas de negocio sobre fraude, comportamiento temporal, comercios top y cambios fuertes en el gasto de usuarios.

La diferencia principal frente a un `groupBy` tradicional es que las window functions permiten conservar el detalle de las filas mientras se calculan métricas por grupo o por secuencia temporal.